# RAG Evaluation Pipeline
Αξιολόγηση του RAG συστήματος με RAGAS + Excel

**Βήμα 1:** Φόρτωση ερωτήσεων/απαντήσεων από Excel  
**Βήμα 2:** Εκτέλεση RAG pipeline για κάθε ερώτηση  
**Βήμα 3:** Αξιολόγηση με RAGAS metrics  
**Βήμα 4:** Αποθήκευση αποτελεσμάτων

## Cell 1: Εγκατάσταση βιβλιοθηκών

In [1]:
!pip install ragas openpyxl pandas datasets --quiet


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## Cell 2: Imports & Config

In [ ]:
import sys
sys.path.insert(0, ".")

import json
import pandas as pd
from pathlib import Path
from datasets import Dataset
from datasets import load_dataset

from rag_retriever_base import RAGRetriever
from config import (
    EVAL_QUESTIONS_PATH,
    EVAL_RESULTS_PATH,
    EVAL_INCREMENTAL_PATH,
)

EXCEL_PATH       = EVAL_QUESTIONS_PATH
RESULTS_PATH     = EVAL_RESULTS_PATH
INCREMENTAL_PATH = EVAL_INCREMENTAL_PATH

retriever = RAGRetriever()
print(f"Imports OK! Collection: {retriever.count()} documents")


c:\Users\user\Documents\Advanced-Rag-for-industrial-manuals-main\Advanced-Rag-for-industrial-manuals-main\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading embedding model...
Ready. Collection has 25022 documents.
Imports OK! Collection: 25022 documents


## Cell 3: Δομή του Excel

Το Excel πρέπει να έχει αυτές τις στήλες:

| Question | Answer |
|----------|-------------|
| What is the max spindle speed? | The maximum spindle speed is 24000 RPM |
| What are the safety requirements? | ... |

In [3]:
# Φόρτωση ερωτήσεων από Excel
df = pd.read_excel(EXCEL_PATH)

# Έλεγχος στηλών
required_cols = ["Question", "Answer"]
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f"Λείπει η στήλη '{col}' από το Excel!")

print(f"Βρέθηκαν {len(df)} ερωτήσεις")
print("\nΠρώτες 5 ερωτήσεις:")
print(df[["Question", "Answer"]].head(5).to_string())

Βρέθηκαν 120 ερωτήσεις

Πρώτες 5 ερωτήσεις:
                                                                                                                                                                                   Question                                                                                    Answer
0                         According to TruLaser-2030-Pre-Install-Manual, what are the required material specification and sheet thickness for the workpieces needed for machine acceptance?  Material DD11 Mat. No. 1.0332 (EN 10111) with a sheet thickness of s = 2.0 mm / 0.08 in.
1  According to TruLaser-2030-Pre-Install-Manual, what is the minimum required kVA power rating for an isolating transformer if it must be higher than the machine nameplate specification?                                                                       At least 10% higher
2                                                                                             According to TruLaser-2030-P

## Cell 4: Φόρτωση Vector DB και Embedding Model

In [ ]:
import importlib
import rag_retriever
importlib.reload(rag_retriever)
from rag_retriever_v3 import RAGRetriever
retriever = RAGRetriever()


In [ ]:
# Το retrieve και generate_answer γίνονται πλέον μέσω RAGRetriever
# Test:
# Επιλογή 1: πραγματική ερώτηση
docs   = retriever.retrieve("What is the maximum spindle speed?", n_results=5)
answer = retriever.generate_answer("What is the maximum spindle speed?", docs)
print(f"Pipeline OK — {len(docs)} docs retrieved")



## Cell 5: Εκτέλεση RAG για κάθε ερώτηση

In [4]:
# Φόρτωση ήδη επεξεργασμένων — συνέχεια από crash
import os
if INCREMENTAL_PATH.exists():
    os.remove(INCREMENTAL_PATH)
    print("Deleted — θα ξαναρχίσει από το 0")

if INCREMENTAL_PATH.exists():
    with open(INCREMENTAL_PATH, "r", encoding="utf-8") as f:
        saved = json.load(f)
    questions         = saved["questions"]
    ground_truths     = saved["ground_truths"]
    answers           = saved["answers"]
    contexts_list     = saved["contexts_list"]
    processed_indices = set(saved.get("processed_indices", []))
    print(f"Συνέχεια από {len(questions)} ήδη επεξεργασμένες ερωτήσεις...")
else:
    questions = []; ground_truths = []; answers = []; contexts_list = []
    processed_indices = set()

for idx, row in df.iterrows():
    if idx in processed_indices:
        print(f"[{idx+1}/{len(df)}] παρακάμπτεται (ήδη done)")
        continue

    question     = row["Question"]
    ground_truth = row["Answer"]
    print(f"[{idx+1}/{len(df)}] {question[:60]}...")

    docs     = retriever.retrieve(question, n_results=5)
    contexts = [doc["text"] for doc in docs if doc.get("text")]
    answer   = retriever.generate_answer(question, docs)

    questions.append(question); ground_truths.append(ground_truth)
    answers.append(answer);     contexts_list.append(contexts)
    processed_indices.add(idx)
    print(f"  → {answer[:80]}...")

    INCREMENTAL_PATH.write_text(
        json.dumps({"questions": questions, "ground_truths": ground_truths,
                    "answers": answers, "contexts_list": contexts_list,
                    "processed_indices": list(processed_indices)},
                   ensure_ascii=False, indent=2), encoding="utf-8"
    )

print(f"\nΟλοκληρώθηκε! {len(answers)} απαντήσεις.")


Deleted — θα ξαναρχίσει από το 0
[1/120] According to TruLaser-2030-Pre-Install-Manual, what are the ...
  → The customer must provide two sheets of DD11 Mat. No. 1.0332 (EN 10111) with a d...
[2/120] According to TruLaser-2030-Pre-Install-Manual, what is the m...
  → The kVA power of the transformer must be at least 10% higher than specified on t...
[3/120] According to TruLaser-2030-Pre-Install-Manual, who must be e...
  → The provided text does not specify which personnel or entities must be employed ...
[4/120] According to TruLaser-2030-Pre-Install-Manual, what is the m...
  → The provided context does not contain information regarding the meanings of warn...
[5/120] According to 280zy0uxx5okr0va, what is the range for rapid t...
  → The provided context does not contain information regarding "280zy0uxx5okr0va" o...
[6/120] According to 280zy0uxx5okr0va, what are the maximum speed (i...
  → The provided text from source 280zy0uxx5okr0va does not contain any information ...
[7/120]

In [6]:
import os
os.environ["OPENAI_API_KEY"] = "YOUR_OPENAI_API_KEY"  # your real key


In [ ]:
import requests
resp = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "qwen3:8b",
        "prompt": "What is 2+2?",
        "stream": False,
        "think": True,
        "options": {"num_ctx": 512}
    },
    timeout=300
)
data = resp.json()
print("thinking:", data.get("thinking", "NOT PRESENT"))
print("response:", data.get("response"))


## Cell 6: Αξιολόγηση με RAGAS

In [5]:
!pip install langchain-huggingface --quiet


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import time, requests

start = time.time()
resp = requests.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "qwen3:8b",
        "system": "You are a helpful assistant.",
        "prompt": "What is 2+2? Answer in one word.",
        "stream": False,
        "think": False,
        "options": {"num_ctx": 512, "temperature": 0.1, "num_gpu": 99}
    },
    timeout=300
)
elapsed = time.time() - start
data = resp.json()
print(f"Time: {elapsed:.1f}s")
print(f"Response: {data.get('response')}")
print(f"Think field present: {'thinking' in data}")


In [7]:
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision, answer_correctness,answer_similarity
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_community.embeddings import HuggingFaceEmbeddings
from ragas import RunConfig
import os
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]


run_config = RunConfig(timeout=120, max_retries=2, max_workers=4)

truncated_contexts = [[c[:600] for c in ctx[:2]] for ctx in contexts_list]

eval_dataset = Dataset.from_dict({
    "question":     questions,
    "answer":       answers,
    "contexts":     truncated_contexts,
    "ground_truth": ground_truths
})
print(f"Eval dataset: {len(eval_dataset)} questions")

# GPT-4o as judge
gpt4o      = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)
ragas_llm  = LangchainLLMWrapper(gpt4o)

# Same embedding model as the vector DB
hf_embeddings    = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")
ragas_embeddings = LangchainEmbeddingsWrapper(hf_embeddings)

results = evaluate(
    dataset=eval_dataset,
    metrics=[faithfulness, answer_relevancy, context_recall, context_precision, answer_correctness, answer_similarity],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
    run_config=run_config
)

print("\n=== RAGAS RESULTS ===")
print(f"Faithfulness:       {results['faithfulness']:.3f}")
print(f"Answer Relevancy:   {results['answer_relevancy']:.3f}")
print(f"Context Recall:     {results['context_recall']:.3f}")
print(f"Context Precision:  {results['context_precision']:.3f}")
print(f"Answer Correctness: {results['answer_correctness']:.3f}")
print(f"Answer Similarity:  {results['answer_similarity']:.3f}")

C:\Users\user\AppData\Local\Temp\ipykernel_6868\2333716328.py:2: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision, answer_correctness,answer_similarity
C:\Users\user\AppData\Local\Temp\ipykernel_6868\2333716328.py:2: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision, answer_correctness,answer_similarity
C:\Users\user\AppData\Local\Temp\ipykernel_6868\2333716328.py:2: DeprecationWarning: Importing context_recall from 'ragas.metrics' is deprecated and will be removed in 

Eval dataset: 120 questions


C:\Users\user\AppData\Local\Temp\ipykernel_6868\2333716328.py:26: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm  = LangchainLLMWrapper(gpt4o)
C:\Users\user\AppData\Local\Temp\ipykernel_6868\2333716328.py:29: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  hf_embeddings    = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5")
C:\Users\user\AppData\Local\Temp\ipykernel_6868\2333716328.py:30: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be rem


=== RAGAS RESULTS ===


TypeError: unsupported format string passed to list.__format__

## Cell 7: Αποθήκευση αποτελεσμάτων στο Excel

In [8]:
# Δημιουργία DataFrame με αποτελέσματα
results_df = results.to_pandas()

# Προσθήκη ερωτήσεων και απαντήσεων
results_df.insert(0, "question",     questions)
results_df.insert(1, "ground_truth", ground_truths)
results_df.insert(2, "rag_answer",   answers)

# Αποθήκευση σε Excel
results_df.to_excel(RESULTS_PATH, index=False)
print(f"Αποτελέσματα αποθηκεύτηκαν: {RESULTS_PATH}")

# Συνοπτική εμφάνιση
print("\n=== ΣΥΝΟΨΗ ===")
print(results_df[["question", "faithfulness", "answer_relevancy", 
                   "context_recall", "context_precision"]].to_string())

Αποτελέσματα αποθηκεύτηκαν: c:\Users\user\Documents\Advanced-Rag-for-industrial-manuals-main\Advanced-Rag-for-industrial-manuals-main\data\evaluation_results.xls

=== ΣΥΝΟΨΗ ===
                                                                                                                                                                                                                                         question  faithfulness  answer_relevancy  context_recall  context_precision
0                                                                               According to TruLaser-2030-Pre-Install-Manual, what are the required material specification and sheet thickness for the workpieces needed for machine acceptance?      1.000000          0.785860             1.0                1.0
1                                                        According to TruLaser-2030-Pre-Install-Manual, what is the minimum required kVA power rating for an isolating transformer if it must be higher than

In [ ]:
import json
with open(INCREMENTAL_PATH, "r", encoding="utf-8") as f:
    saved = json.load(f)

for i, ctx in enumerate(saved["contexts_list"][:3]):
    print(f"Q{i+1}: {len(ctx)} chunks, total chars: {sum(len(c) for c in ctx)}")

